# Post-hoc set 3 — Qwen3-VL embedding (H4) and visual reranker (H1) on ViDoRe DocVQA (Colab)

Thin wrapper around `scripts/run_posthoc3_gpu.py` with `configs/posthoc3_docvqa.yaml` (pre-registered in `e70f2a7`).
No metric is computed here; evaluation runs on CPU (dev first, test once).

**Before running:** `Runtime → Change runtime type → L4 GPU`.

In [ ]:
!nvidia-smi

In [ ]:
# Idempotent: clones on first run, pulls the latest commit when the runtime is reused.
import os
if not os.path.isdir("/content/colpali-hybrid-retrieval"):
    !git clone https://github.com/ismailucrn/colpali-hybrid-retrieval.git /content/colpali-hybrid-retrieval
%cd /content/colpali-hybrid-retrieval
!git pull -q
!git log -1 --format='%H %s'

In [ ]:
# Colab preinstalls torchao, which peft rejects at import time; this project does not use it.
!pip uninstall -y -q torchao
!pip install -q -r requirements.txt qwen-vl-utils==0.0.14

In [ ]:
# Smoke tests (never reported): a few queries/pages, separate _smoke output folders.
!python scripts/run_posthoc3_gpu.py --stage embed --limit 8
!python scripts/run_posthoc3_gpu.py --stage rerank --limit 2

In [ ]:
!python scripts/run_posthoc3_gpu.py --stage embed

In [ ]:
!python scripts/run_posthoc3_gpu.py --stage rerank

In [ ]:
!pip freeze > results/qwen3vl_embedding_docvqa/pip_freeze_colab.txt
!ls -la results/qwen3vl_embedding_docvqa
!cat results/qwen3vl_embedding_docvqa/metrics.json
!ls -la results/visual_reranker_docvqa
!cat results/visual_reranker_docvqa/metrics.json

In [ ]:
# Zip the outputs and download them (binary files cannot be copied from the page).
# Guard: "Run all" continues after a failed shell command, so only download a complete run.
import os
for d in ['results/qwen3vl_embedding_docvqa', 'results/visual_reranker_docvqa']:
    assert os.path.exists(d + "/metrics.json"), d + " did not finish; nothing to download"
!zip -q -r /content/posthoc3_qwen3vl.zip results/qwen3vl_embedding_docvqa results/visual_reranker_docvqa
!sha256sum /content/posthoc3_qwen3vl.zip
from google.colab import files
files.download("/content/posthoc3_qwen3vl.zip")